# Titanic Survival Prediction - ML Pipeline

This notebook builds a complete machine learning pipeline to predict Titanic survival rates.

## Pipeline Overview:
1. Data Loading & Exploration
2. Data Preprocessing
3. Feature Engineering
4. Model Building & Training
5. Model Evaluation
6. Hyperparameter Tuning
7. Final Predictions

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score, roc_curve
import warnings
warnings.filterwarnings('ignore')

# Set style for visualizations
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")

## 2. Load and Explore Data

In [ ]:
# Load the dataset
df = pd.read_csv('data/train.csv')

# Display basic information
print("Dataset Shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nDataset Info:")
print(df.info())
print("\nMissing Values:")
print(df.isnull().sum())
print("\nBasic Statistics:")
print(df.describe())

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Survival distribution
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Survival count
sns.countplot(data=df, x='Survived', ax=axes[0, 0])
axes[0, 0].set_title('Survival Distribution')

# Survival by Gender
sns.countplot(data=df, x='Sex', hue='Survived', ax=axes[0, 1])
axes[0, 1].set_title('Survival by Gender')

# Survival by Passenger Class
sns.countplot(data=df, x='Pclass', hue='Survived', ax=axes[1, 0])
axes[1, 0].set_title('Survival by Passenger Class')

# Age distribution by Survival
df[df['Survived'] == 0]['Age'].hist(bins=30, ax=axes[1, 1], alpha=0.5, label='Did Not Survive')
df[df['Survived'] == 1]['Age'].hist(bins=30, ax=axes[1, 1], alpha=0.5, label='Survived')
axes[1, 1].set_title('Age Distribution by Survival')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

print("\nSurvival Rate:", df['Survived'].mean())
print("\nSurvival by Gender:")
print(df.groupby('Sex')['Survived'].mean())
print("\nSurvival by Passenger Class:")
print(df.groupby('Pclass')['Survived'].mean())

## 4. Data Preprocessing

In [ ]:
# Create a copy for preprocessing
df_processed = df.copy()

# Handle missing Age values - fill with median
df_processed['Age'].fillna(df_processed['Age'].median(), inplace=True)

# Handle missing Embarked values - fill with mode
df_processed['Embarked'].fillna(df_processed['Embarked'].mode()[0], inplace=True)

# Drop Cabin column (too many missing values)
df_processed.drop('Cabin', axis=1, inplace=True)

# Drop PassengerId, Name, and Ticket (not useful for prediction)
df_processed.drop(['PassengerId', 'Name', 'Ticket'], axis=1, inplace=True)

print("Missing values after preprocessing:")
print(df_processed.isnull().sum())
print("\nDataset shape:", df_processed.shape)

## 5. Feature Engineering

In [ ]:
# Create new features

# Family Size
df_processed['FamilySize'] = df_processed['SibSp'] + df_processed['Parch'] + 1

# Is Alone
df_processed['IsAlone'] = (df_processed['FamilySize'] == 1).astype(int)

# Age Groups
df_processed['AgeGroup'] = pd.cut(df_processed['Age'], bins=[0, 12, 18, 35, 60, 100], 
                                    labels=['Child', 'Teen', 'Adult', 'Middle-aged', 'Senior'])

# Fare per person
df_processed['FarePerPerson'] = df_processed['Fare'] / df_processed['FamilySize']

print("New features created!")
print("\nDataframe with new features:")
print(df_processed.head())

## 6. Encode Categorical Variables

In [ ]:
# Encode Sex
df_processed['Sex'] = df_processed['Sex'].map({'male': 1, 'female': 0})

# Encode Embarked
embarked_mapping = {'S': 0, 'C': 1, 'Q': 2}
df_processed['Embarked'] = df_processed['Embarked'].map(embarked_mapping)

# Encode AgeGroup
age_group_mapping = {'Child': 0, 'Teen': 1, 'Adult': 2, 'Middle-aged': 3, 'Senior': 4}
df_processed['AgeGroup'] = df_processed['AgeGroup'].map(age_group_mapping)

# Drop SibSp and Parch as we have FamilySize
df_processed.drop(['SibSp', 'Parch'], axis=1, inplace=True)

print("Categorical variables encoded!")
print("\nProcessed dataset:")
print(df_processed.head())
print("\nFinal dataset info:")
print(df_processed.info())

## 7. Prepare Features and Target

In [ ]:
# Separate features and target
X = df_processed.drop('Survived', axis=1)
y = df_processed['Survived']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set size: {X_train.shape}")
print(f"Testing set size: {X_test.shape}")
print(f"Features: {X.columns.tolist()}")

## 8. Train Multiple Models

In [ ]:
# Initialize models
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(kernel='rbf', probability=True, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}

# Dictionary to store results
results = {}

# Train and evaluate models
for name, model in models.items():
    # Use scaled features for all models
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, model.predict_proba(X_test_scaled)[:, 1])
    
    results[name] = {
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'ROC-AUC': roc_auc
    }
    
    print(f"\n{name} Results:")
    print(f"  Accuracy:  {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"  ROC-AUC:   {roc_auc:.4f}")

## 9. Compare Model Performance

In [ ]:
# Create results dataframe
results_df = pd.DataFrame(results).T
print("Model Comparison:")
print(results_df)

# Visualize results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy comparison
results_df['Accuracy'].plot(kind='barh', ax=axes[0], color='skyblue')
axes[0].set_title('Model Accuracy Comparison')
axes[0].set_xlabel('Accuracy')

# All metrics comparison
results_df.plot(kind='barh', ax=axes[1])
axes[1].set_title('All Metrics Comparison')
axes[1].set_xlabel('Score')

plt.tight_layout()
plt.show()

# Find best model
best_model_name = results_df['Accuracy'].idxmax()
print(f"\nBest Model: {best_model_name} with {results_df['Accuracy'].max():.4f} accuracy")

## 10. Detailed Analysis of Best Model

In [ ]:
# Get predictions from best model
best_model = models[best_model_name]
y_pred = best_model.predict(X_test_scaled)
y_pred_proba = best_model.predict_proba(X_test_scaled)[:, 1]

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Did Not Survive', 'Survived']))

## 11. Visualize Confusion Matrix and ROC Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], 
            xticklabels=['Did Not Survive', 'Survived'],
            yticklabels=['Did Not Survive', 'Survived'])
axes[0].set_title(f'Confusion Matrix - {best_model_name}')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
auc = roc_auc_score(y_test, y_pred_proba)
axes[1].plot(fpr, tpr, label=f'ROC Curve (AUC = {auc:.4f})', linewidth=2)
axes[1].plot([0, 1], [0, 1], 'k--', label='Random Classifier')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title(f'ROC Curve - {best_model_name}')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 12. Cross-Validation

In [ ]:
# Perform cross-validation on the best model
cv_scores = cross_val_score(best_model, X_train_scaled, y_train, cv=5, scoring='accuracy')

print(f"Cross-Validation Scores ({best_model_name}):")
print(f"  Fold scores: {cv_scores}")
print(f"  Mean accuracy: {cv_scores.mean():.4f}")
print(f"  Standard deviation: {cv_scores.std():.4f}")

# Visualize cross-validation scores
plt.figure(figsize=(10, 5))
plt.plot(range(1, len(cv_scores) + 1), cv_scores, 'o-', linewidth=2, markersize=8)
plt.axhline(y=cv_scores.mean(), color='r', linestyle='--', label=f'Mean: {cv_scores.mean():.4f}')
plt.xlabel('Fold')
plt.ylabel('Accuracy')
plt.title(f'Cross-Validation Scores - {best_model_name}')
plt.xticks(range(1, len(cv_scores) + 1))
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 13. Feature Importance (for tree-based models)

In [ ]:
# Check if model has feature importance
if hasattr(best_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'Feature': X.columns,
        'Importance': best_model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    print("Feature Importance:")
    print(feature_importance)
    
    # Visualize feature importance
    plt.figure(figsize=(10, 6))
    plt.barh(feature_importance['Feature'], feature_importance['Importance'], color='steelblue')
    plt.xlabel('Importance')
    plt.title(f'Feature Importance - {best_model_name}')
    plt.tight_layout()
    plt.show()
else:
    print(f"\n{best_model_name} does not have feature importance attribute.")

## 14. Hyperparameter Tuning (Optional - Example with Random Forest)

In [ ]:
# Hyperparameter tuning for Random Forest
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

print("Performing Grid Search for Random Forest...")
print("This may take a minute...\n")

rf_grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, scoring='accuracy', n_jobs=-1)
rf_grid.fit(X_train_scaled, y_train)

print(f"Best parameters: {rf_grid.best_params_}")
print(f"Best cross-validation score: {rf_grid.best_score_:.4f}")

# Evaluate tuned model
y_pred_tuned = rf_grid.predict(X_test_scaled)
tuned_accuracy = accuracy_score(y_test, y_pred_tuned)
print(f"\nTuned Random Forest Test Accuracy: {tuned_accuracy:.4f}")

## 15. Summary and Conclusions

In [ ]:
print("="*60)
print("TITANIC SURVIVAL PREDICTION - FINAL SUMMARY")
print("="*60)
print(f"\nBest Model: {best_model_name}")
print(f"Test Set Accuracy: {results[best_model_name]['Accuracy']:.4f}")
print(f"ROC-AUC Score: {results[best_model_name]['ROC-AUC']:.4f}")
print(f"\nAll Models Performance:")
print(results_df.to_string())
print("\n" + "="*60)
print("Key Insights:")
print("="*60)
print("1. Gender was a strong predictor of survival (females had higher survival rates)")
print("2. Passenger class significantly affected survival chances (1st class had highest survival)")
print("3. Family size and being alone influenced survival outcomes")
print("4. Younger passengers had better survival chances")
print(f"5. The {best_model_name} model performed best among all tested models")
print("\n" + "="*60)